<a href="https://colab.research.google.com/github/kasrasa/ViT-VLM-experiments/blob/ViT-Experiments/ViT_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch
!pip install -q timm
!pip install -q evaluate
!pip install -q peft
!pip install --upgrade -q torchao
!pip install -q scikit-learn
!pip install -q matplotlib seaborn accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.2 MB/s eta 0:00:00


In [2]:
import os

# Set to True to always fine-tune, False to load existing checkpoints if available
FORCE_FINE_TUNING = True

In [3]:
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()

In [4]:
import evaluate
accuracy = evaluate.load("accuracy")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from datasets import load_dataset, DatasetDict
from collections import defaultdict, Counter
import random
import numpy as np

# -----------------------------
# Balanced Food101 subset config
# -----------------------------
DATASET_NAME = "ethz/food101"
NUM_SELECTED_CLASSES = 20
SAMPLES_PER_CLASS = 250
TEST_SIZE = 0.20
SEED = 42

# Load the full Food101 training split.
# We use the train split and create our own stratified train/validation split
# because this experiment is meant to compare fine-tuning strategies cheaply.
food_full_train = load_dataset(DATASET_NAME, split="train")
original_labels = food_full_train.features["label"].names

rng = random.Random(SEED)

# Group dataset indices by original Food101 class id
label_to_indices = defaultdict(list)
for idx, label_id in enumerate(food_full_train["label"]):
    label_to_indices[int(label_id)].append(idx)

# Keep only classes that have enough examples
eligible_label_ids = [
    label_id
    for label_id, indices in label_to_indices.items()
    if len(indices) >= SAMPLES_PER_CLASS
]

if len(eligible_label_ids) < NUM_SELECTED_CLASSES:
    raise ValueError(
        f"Only {len(eligible_label_ids)} classes have at least "
        f"{SAMPLES_PER_CLASS} samples. Need {NUM_SELECTED_CLASSES}."
    )

# Select 20 classes reproducibly
selected_old_label_ids = sorted(rng.sample(eligible_label_ids, NUM_SELECTED_CLASSES))

# Select exactly 250 images per selected class
selected_indices = []
for old_label_id in selected_old_label_ids:
    indices = label_to_indices[old_label_id].copy()
    rng.shuffle(indices)
    selected_indices.extend(indices[:SAMPLES_PER_CLASS])

rng.shuffle(selected_indices)

food_subset = food_full_train.select(selected_indices)

print(f"Total selected images: {len(food_subset)}")
print(f"Selected classes: {len(selected_old_label_ids)}")
print("Selected class names:")
for old_label_id in selected_old_label_ids:
    print(f"  old_id={old_label_id:3d} -> {original_labels[old_label_id]}")

# Stratified split while labels are still original Food101 label IDs
food = food_subset.train_test_split(
    test_size=TEST_SIZE,
    shuffle=True,
    seed=SEED,
    stratify_by_column="label",
)

# Remap selected original labels to contiguous labels 0..19.
# This is important because the classifier head will have exactly 20 outputs.
old_to_new = {
    old_label_id: new_label_id
    for new_label_id, old_label_id in enumerate(selected_old_label_ids)
}

new_to_old = {
    new_label_id: old_label_id
    for old_label_id, new_label_id in old_to_new.items()
}

labels = [
    original_labels[new_to_old[new_label_id]]
    for new_label_id in range(NUM_SELECTED_CLASSES)
]

id2label = {
    new_label_id: label_name
    for new_label_id, label_name in enumerate(labels)
}

label2id = {
    label_name: new_label_id
    for new_label_id, label_name in id2label.items()
}

def remap_label(example):
    example["label"] = old_to_new[int(example["label"])]
    return example

food = DatasetDict({
    "train": food["train"].map(remap_label),
    "test": food["test"].map(remap_label),
})

# Sanity checks
train_counts = Counter(food["train"]["label"])
test_counts = Counter(food["test"]["label"])

print("\nAfter remapping:")
print(f"Train size: {len(food['train'])}")
print(f"Validation size: {len(food['test'])}")
print(f"Number of labels: {len(labels)}")
print(f"Train class counts: {sorted(train_counts.items())}")
print(f"Validation class counts: {sorted(test_counts.items())}")

assert len(food["train"]) + len(food["test"]) == NUM_SELECTED_CLASSES * SAMPLES_PER_CLASS
assert set(train_counts.keys()) == set(range(NUM_SELECTED_CLASSES))
assert set(test_counts.keys()) == set(range(NUM_SELECTED_CLASSES))


README.md:   0%|          | 0.00/16.4k [00:00<?, ?B/s]

data/train-00000-of-00008.parquet:   0%|          | 0.00/490M [00:00<?, ?B/s]

data/train-00001-of-00008.parquet:   0%|          | 0.00/464M [00:00<?, ?B/s]

data/train-00002-of-00008.parquet:   0%|          | 0.00/472M [00:00<?, ?B/s]

data/train-00003-of-00008.parquet:   0%|          | 0.00/464M [00:00<?, ?B/s]

data/train-00004-of-00008.parquet:   0%|          | 0.00/475M [00:00<?, ?B/s]

In [ ]:
# Label mappings for the 20-class balanced Food101 subset.
# These are already created in the dataset-loading cell above.
# The classifier head should use len(labels) == 20.

print(f"Number of selected labels: {len(labels)}")
print("id2label:")
for class_id, class_name in id2label.items():
    print(f"  {class_id}: {class_name}")

print("\nlabel2id:")
for class_name, class_id in label2id.items():
    print(f"  {class_name}: {class_id}")


In [ ]:
from torchvision.transforms import (
    RandomResizedCrop,
    Resize,
    CenterCrop,
    Compose,
    Normalize,
    ToTensor,
)

def get_image_size(image_processor):
    if "shortest_edge" in image_processor.size:
        return image_processor.size["shortest_edge"]
    return image_processor.size["height"]

def apply_transforms(examples, image_processor, is_train=True):
    image_size = get_image_size(image_processor)
    normalize = Normalize(
        mean=image_processor.image_mean,
        std=image_processor.image_std,
    )

    if is_train:
        transform = Compose([
            RandomResizedCrop(image_size),
            ToTensor(),
            normalize,
        ])
    else:
        transform = Compose([
            Resize(image_size),
            CenterCrop(image_size),
            ToTensor(),
            normalize,
        ])

    examples["pixel_values"] = [
        transform(img.convert("RGB")) for img in examples["image"]
    ]
    del examples["image"]
    return examples

## Update `compute_metrics` Function

This section updates the `compute_metrics` function to include the F1-score in addition to accuracy. This provides a more comprehensive evaluation metric, especially for imbalanced datasets.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def softmax_np(logits):
    """
    Numerically stable softmax.
    """
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def expected_calibration_error(probs, labels, n_bins=10):
    """
    Expected Calibration Error.

    If the model says it is 80% confident, we want it to be correct
    about 80% of the time. ECE measures this mismatch.
    """
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    correct = predictions == labels

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lower = bin_edges[i]
        upper = bin_edges[i + 1]

        if i == 0:
            in_bin = (confidences >= lower) & (confidences <= upper)
        else:
            in_bin = (confidences > lower) & (confidences <= upper)

        prop_in_bin = np.mean(in_bin)

        if prop_in_bin > 0:
            avg_confidence = np.mean(confidences[in_bin])
            avg_accuracy = np.mean(correct[in_bin])
            ece += prop_in_bin * abs(avg_confidence - avg_accuracy)

    return float(ece)


def compute_metrics(eval_pred):
    """
    Metrics used during Trainer evaluation.

    Includes:
    - accuracy
    - macro F1
    - weighted F1
    - top-5 accuracy
    - confidence statistics
    - ECE calibration metric
    - NLL
    - multiclass Brier score
    """
    logits = eval_pred.predictions

    # Some Hugging Face models may return predictions as a tuple.
    if isinstance(logits, tuple):
        logits = logits[0]

    labels = eval_pred.label_ids
    predictions = np.argmax(logits, axis=1)
    probs = softmax_np(logits)

    num_classes = probs.shape[1]
    k = min(5, num_classes)

    top_k_predictions = np.argsort(probs, axis=1)[:, -k:]
    top_k_accuracy = np.mean([
        labels[i] in top_k_predictions[i]
        for i in range(len(labels))
    ])

    top1_confidence = np.max(probs, axis=1)

    sorted_probs = np.sort(probs, axis=1)
    top2_confidence = sorted_probs[:, -2] if num_classes > 1 else np.zeros_like(top1_confidence)
    top1_top2_margin = top1_confidence - top2_confidence

    correct_mask = predictions == labels
    wrong_mask = ~correct_mask

    correct_mean_confidence = float(np.mean(top1_confidence[correct_mask])) if np.any(correct_mask) else 0.0
    wrong_mean_confidence = float(np.mean(top1_confidence[wrong_mask])) if np.any(wrong_mask) else 0.0

    # Negative log likelihood for the true class
    eps = 1e-12
    true_class_probs = probs[np.arange(len(labels)), labels]
    nll = -np.mean(np.log(np.clip(true_class_probs, eps, 1.0)))

    # Multiclass Brier score
    one_hot = np.zeros_like(probs)
    one_hot[np.arange(len(labels)), labels] = 1.0
    brier_score = np.mean(np.sum((probs - one_hot) ** 2, axis=1))

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, predictions, average="weighted", zero_division=0),

        # Keep this alias so your existing summary table does not break
        "f1": f1_score(labels, predictions, average="weighted", zero_division=0),

        "top_5_accuracy": float(top_k_accuracy),
        "mean_top1_confidence": float(np.mean(top1_confidence)),
        "mean_top1_top2_margin": float(np.mean(top1_top2_margin)),
        "correct_mean_confidence": correct_mean_confidence,
        "wrong_mean_confidence": wrong_mean_confidence,
        "ece": expected_calibration_error(probs, labels, n_bins=10),
        "nll": float(nll),
        "brier_score": float(brier_score),
    }

## Define Parameter Counting Utility

This section defines a Python function to count and display the number of trainable parameters in a given PyTorch model. This function will be reused at various stages to track model complexity.

This function `count_parameters` iterates through model parameters, sums the elements of those that require gradients, and then prints the result in millions.

In [ ]:
def count_parameters(model):
    """
    Counts and displays the number of trainable parameters in a PyTorch model.
    """
    num_params_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Number of trainable parameters: {num_params_trainable:,} ({num_params_trainable / 1e6:.2f} million)")
    print(f"Total parameters: {total_params:,} ({total_params / 1e6:.2f} million)")


## Define Reusable Evaluation and Plotting Function

This section creates a function that takes a trained model, a dataset, and label mappings as input, computes predictions, calculates accuracy and F1-score, and generates a confusion matrix. This function will be used to evaluate all fine-tuned models on both Food101 and CIFAR-10 datasets.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import pandas as pd

def evaluate_and_plot(
    model,
    trainer,
    dataset,
    id2label_mapping,
    dataset_name,
    num_labels,
    model_display_name,
    training_time=None,
    normalize_cm=False,
):
    print(f"\n--- Evaluating {model_display_name} on {dataset_name} ---")

    predictions_output = trainer.predict(dataset)

    logits = predictions_output.predictions
    if isinstance(logits, tuple):
        logits = logits[0]

    labels = predictions_output.label_ids
    predicted_labels = np.argmax(logits, axis=1)
    probs = softmax_np(logits)

    metrics = compute_metrics(predictions_output)

    # Add loss from Trainer prediction output if available
    eval_loss = predictions_output.metrics.get("test_loss", None)
    if eval_loss is not None:
        metrics["eval_loss"] = eval_loss

    print("\nCore metrics:")
    print(f"Accuracy:                {metrics['accuracy']:.4f}")
    print(f"Macro F1:                {metrics['macro_f1']:.4f}")
    print(f"Weighted F1:             {metrics['weighted_f1']:.4f}")
    print(f"Top-5 Accuracy:          {metrics['top_5_accuracy']:.4f}")

    if eval_loss is not None:
        print(f"Eval Loss:               {eval_loss:.4f}")

    print("\nConfidence / calibration metrics:")
    print(f"Mean Top-1 Confidence:   {metrics['mean_top1_confidence']:.4f}")
    print(f"Mean Top1-Top2 Margin:   {metrics['mean_top1_top2_margin']:.4f}")
    print(f"Correct Mean Confidence: {metrics['correct_mean_confidence']:.4f}")
    print(f"Wrong Mean Confidence:   {metrics['wrong_mean_confidence']:.4f}")
    print(f"ECE:                     {metrics['ece']:.4f}")
    print(f"NLL:                     {metrics['nll']:.4f}")
    print(f"Brier Score:             {metrics['brier_score']:.4f}")

    if training_time is not None:
        print(f"\nTraining time for {model_display_name}: {training_time:.2f} seconds")

    print("\nLabel sanity check:")
    print(f"Number of unique true labels:      {len(np.unique(labels))}")
    print(f"Number of unique predicted labels: {len(np.unique(predicted_labels))}")

    # -----------------------------
    # Confidence analysis dataframe
    # -----------------------------
    top1_conf = np.max(probs, axis=1)
    sorted_probs = np.sort(probs, axis=1)
    top2_conf = sorted_probs[:, -2] if probs.shape[1] > 1 else np.zeros_like(top1_conf)
    margins = top1_conf - top2_conf
    correct = predicted_labels == labels

    confidence_df = pd.DataFrame({
        "true_label_id": labels,
        "pred_label_id": predicted_labels,
        "true_label": [id2label_mapping[int(i)] for i in labels],
        "pred_label": [id2label_mapping[int(i)] for i in predicted_labels],
        "top1_confidence": top1_conf,
        "top2_confidence": top2_conf,
        "top1_top2_margin": margins,
        "correct": correct,
    })

    print("\nConfidence summary:")
    display(
        confidence_df.groupby("correct")[[
            "top1_confidence",
            "top1_top2_margin"
        ]].agg(["mean", "median", "min", "max", "count"]).round(4)
    )

    print("\nMost confident wrong predictions:")
    wrong_predictions = confidence_df[confidence_df["correct"] == False]
    if len(wrong_predictions) > 0:
        display(
            wrong_predictions
            .sort_values("top1_confidence", ascending=False)
            .head(10)
            .round(4)
        )
    else:
        print("No wrong predictions found.")

    print("\nLowest-margin predictions:")
    display(
        confidence_df
        .sort_values("top1_top2_margin", ascending=True)
        .head(10)
        .round(4)
    )

    # -----------------------------
    # Full confusion matrix
    # -----------------------------
    all_label_ids = list(range(num_labels))
    target_names = [id2label_mapping[i] for i in all_label_ids]

    cm = confusion_matrix(
        labels,
        predicted_labels,
        labels=all_label_ids,
    )

    if normalize_cm:
        cm_to_plot = cm.astype("float") / cm.sum(axis=1, keepdims=True)
        cm_to_plot = np.nan_to_num(cm_to_plot)
    else:
        cm_to_plot = cm

    plt.figure(figsize=(12, 10))

    if num_labels > 20:
        sns.heatmap(
            cm_to_plot,
            cmap="Blues",
            xticklabels=False,
            yticklabels=False,
            cbar=True,
        )
        plt.title(
            f"Confusion Matrix for {model_display_name} on {dataset_name}\n"
            f"Labels omitted because there are {num_labels} classes"
        )
    else:
        sns.heatmap(
            cm_to_plot,
            annot=True,
            fmt=".2f" if normalize_cm else "g",
            cmap="Blues",
            xticklabels=target_names,
            yticklabels=target_names,
        )
        plt.title(f"Confusion Matrix for {model_display_name} on {dataset_name}")

    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.show()

    # -----------------------------
    # Classification report
    # -----------------------------
    print(f"\n--- Per-class metrics for {model_display_name} ---")

    report = classification_report(
        labels,
        predicted_labels,
        labels=all_label_ids,
        target_names=target_names,
        output_dict=True,
        zero_division=0,
    )

    report_df = pd.DataFrame(report).transpose()

    class_metrics_df = report_df.loc[target_names].copy()
    class_metrics_df["f1-score"] = pd.to_numeric(class_metrics_df["f1-score"], errors="coerce")

    print("\nTop 10 weakest classes by F1-score:")
    weakest_classes = class_metrics_df.sort_values("f1-score", ascending=True).head(10)
    display(weakest_classes[["precision", "recall", "f1-score", "support"]].round(4))

    print("\nTop 10 strongest classes by F1-score:")
    strongest_classes = class_metrics_df.sort_values("f1-score", ascending=False).head(10)
    display(strongest_classes[["precision", "recall", "f1-score", "support"]].round(4))

    # -----------------------------
    # Refined confusion matrix for weakest classes
    # -----------------------------
    weakest_class_names = weakest_classes.index.tolist()
    weakest_label_ids = [target_names.index(name) for name in weakest_class_names]

    if len(weakest_label_ids) > 0:
        mask = np.isin(labels, weakest_label_ids)
        filtered_true = labels[mask]
        filtered_pred = predicted_labels[mask]

        if len(filtered_true) > 0:
            refined_cm = confusion_matrix(
                filtered_true,
                filtered_pred,
                labels=weakest_label_ids,
            )

            if normalize_cm:
                refined_cm_to_plot = refined_cm.astype("float") / refined_cm.sum(axis=1, keepdims=True)
                refined_cm_to_plot = np.nan_to_num(refined_cm_to_plot)
            else:
                refined_cm_to_plot = refined_cm

            plt.figure(figsize=(10, 8))
            sns.heatmap(
                refined_cm_to_plot,
                annot=True,
                fmt=".2f" if normalize_cm else "g",
                cmap="Reds",
                xticklabels=weakest_class_names,
                yticklabels=weakest_class_names,
            )
            plt.title(
                f"Refined Confusion Matrix for Weakest Classes\n"
                f"{model_display_name} on {dataset_name}"
            )
            plt.xlabel("Predicted label")
            plt.ylabel("True label")
            plt.tight_layout()
            plt.show()
        else:
            print("No samples found for weakest classes.")

    # -----------------------------
    # Plotting Train vs. Validation Loss
    # -----------------------------
    train_losses_plot = []
    eval_losses_plot = []
    train_steps_plot = []
    eval_steps_plot = []

    # Get log history from the trainer
    log_history = trainer.state.log_history

    for entry in log_history:
        if 'loss' in entry and 'step' in entry:
            train_losses_plot.append(entry['loss'])
            train_steps_plot.append(entry['step'])
        if 'eval_loss' in entry and 'step' in entry:
            eval_losses_plot.append(entry['eval_loss'])
            eval_steps_plot.append(entry['step'])

    if train_steps_plot or eval_steps_plot:
        plt.figure(figsize=(10, 6))
        if train_steps_plot:
            plt.plot(train_steps_plot, train_losses_plot, label='Training Loss', marker='o', linestyle='-', markersize=4)
        if eval_steps_plot:
            plt.plot(eval_steps_plot, eval_losses_plot, label='Validation Loss', marker='x', linestyle='--', markersize=4)

        plt.title(f'{model_display_name} - Training vs. Validation Loss Over Steps')
        plt.xlabel('Training Steps')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()
    else:
        print(f"No loss history found for {model_display_name} to plot.")


    metrics["classification_report_df"] = report_df
    metrics["confidence_df"] = confidence_df

    return metrics

In [ ]:
from transformers.trainer_utils import get_last_checkpoint
import os
import json
import torch
from safetensors.torch import load_file

try:
    from peft import set_peft_model_state_dict
except Exception:
    set_peft_model_state_dict = None


def has_model_weights(path):
    """
    Checks whether a directory contains saved model weights.
    Supports regular HF model weights and PEFT/LoRA adapter weights.
    """
    if path is None or not os.path.isdir(path):
        return False

    possible_files = [
        "pytorch_model.bin",
        "model.safetensors",
        "adapter_model.bin",
        "adapter_model.safetensors",
    ]

    return any(os.path.exists(os.path.join(path, f)) for f in possible_files)


def load_weights_into_existing_model(model_instance, checkpoint_path, model_name_for_log):
    """
    Loads saved weights into the existing model object.

    This is safer than only detecting that a model exists and then skipping training,
    because we need the current notebook model object to actually contain the saved weights.
    """
    print(f"Loading saved weights for {model_name_for_log} from: {checkpoint_path}")

    # Case 1: PEFT / LoRA adapter checkpoint
    adapter_safetensors = os.path.join(checkpoint_path, "adapter_model.safetensors")
    adapter_bin = os.path.join(checkpoint_path, "adapter_model.bin")

    if os.path.exists(adapter_safetensors) or os.path.exists(adapter_bin):
        if set_peft_model_state_dict is None:
            raise ImportError(
                "PEFT adapter weights found, but `set_peft_model_state_dict` could not be imported."
            )

        if os.path.exists(adapter_safetensors):
            state_dict = load_file(adapter_safetensors)
        else:
            state_dict = torch.load(adapter_bin, map_location="cpu")

        set_peft_model_state_dict(model_instance, state_dict)
        print(f"Loaded PEFT/LoRA adapter weights for {model_name_for_log}.")
        return True

    # Case 2: regular Hugging Face / PyTorch checkpoint
    model_safetensors = os.path.join(checkpoint_path, "model.safetensors")
    model_bin = os.path.join(checkpoint_path, "pytorch_model.bin")

    if os.path.exists(model_safetensors):
        state_dict = load_file(model_safetensors)
    elif os.path.exists(model_bin):
        state_dict = torch.load(model_bin, map_location="cpu")
    else:
        raise FileNotFoundError(f"No supported model weight file found in {checkpoint_path}")

    load_result = model_instance.load_state_dict(state_dict, strict=False)

    missing_keys = list(load_result.missing_keys)
    unexpected_keys = list(load_result.unexpected_keys)

    if len(missing_keys) > 0:
        print(f"Warning: missing keys while loading {model_name_for_log}:")
        print(missing_keys[:20])
        if len(missing_keys) > 20:
            print(f"... and {len(missing_keys) - 20} more")

    if len(unexpected_keys) > 0:
        print(f"Warning: unexpected keys while loading {model_name_for_log}:")
        print(unexpected_keys[:20])
        if len(unexpected_keys) > 20:
            print(f"... and {len(unexpected_keys) - 20} more")

    print(f"Loaded regular model weights for {model_name_for_log}.")
    return True


def conditional_train_model(model_instance, trainer_instance, training_args_instance, model_name_for_log):
    """
    Trains, resumes, or loads a model depending on checkpoint availability.

    Behavior:
    - If FORCE_FINE_TUNING=True:
        Start training from the current model initialization.
    - If FORCE_FINE_TUNING=False and a final model exists in output_dir:
        Load the final saved model weights into the current model and skip training.
    - If FORCE_FINE_TUNING=False and only a checkpoint-* folder exists:
        Resume training from the latest checkpoint.
    - After training:
        Save the final/best model to output_dir for future reuse.

    Returns:
    - train_runtime as a float, so your existing notebook summary code still works.
    """
    checkpoint_dir = training_args_instance.output_dir
    os.makedirs(checkpoint_dir, exist_ok=True)

    latest_checkpoint = get_last_checkpoint(checkpoint_dir) if os.path.isdir(checkpoint_dir) else None

    final_model_exists = has_model_weights(checkpoint_dir)
    latest_checkpoint_exists = has_model_weights(latest_checkpoint)

    print(f"\n--- Conditional training check for {model_name_for_log} ---")
    print(f"Output directory: {checkpoint_dir}")
    print(f"FORCE_FINE_TUNING: {FORCE_FINE_TUNING}")
    print(f"Final model exists at root: {final_model_exists}")
    print(f"Latest checkpoint: {latest_checkpoint}")
    print(f"Latest checkpoint has weights: {latest_checkpoint_exists}")

    train_runtime = 0.0

    # Case 1: Load final saved model and skip training
    if final_model_exists and not FORCE_FINE_TUNING:
        try:
            load_weights_into_existing_model(
                model_instance=model_instance,
                checkpoint_path=checkpoint_dir,
                model_name_for_log=model_name_for_log,
            )
            trainer_instance.model = model_instance
            print(f"Skipping training for {model_name_for_log}; loaded final saved model.")
            return train_runtime
        except Exception as e:
            print(f"Could not load final saved model for {model_name_for_log}: {e}")
            print("Will try to resume from latest checkpoint if available, otherwise train from scratch.")

    # Case 2: Force fresh training
    if FORCE_FINE_TUNING:
        print(f"Starting fresh fine-tuning for {model_name_for_log}.")
        train_result = trainer_instance.train(resume_from_checkpoint=None)

    # Case 3: Resume from checkpoint
    elif latest_checkpoint_exists:
        print(f"Resuming training for {model_name_for_log} from: {latest_checkpoint}")
        train_result = trainer_instance.train(resume_from_checkpoint=latest_checkpoint)

    # Case 4: No saved weights found
    else:
        print(f"No usable saved model/checkpoint found for {model_name_for_log}. Starting training.")
        train_result = trainer_instance.train()

    train_runtime = float(train_result.metrics.get("train_runtime", 0.0))

    print(f"Training complete for {model_name_for_log}.")
    print(f"Train runtime: {train_runtime:.2f} seconds")

    # Save final/best model at the root output_dir.
    # With load_best_model_at_end=True, trainer.model should already be the best model.
    print(f"Saving final model for {model_name_for_log} to: {checkpoint_dir}")
    trainer_instance.save_model(checkpoint_dir)

    # Save training metrics for reference.
    metrics_path = os.path.join(checkpoint_dir, "train_metrics.json")
    try:
        with open(metrics_path, "w") as f:
            json.dump(train_result.metrics, f, indent=2)
        print(f"Saved train metrics to: {metrics_path}")
    except Exception as e:
        print(f"Could not save train metrics JSON: {e}")

    return train_runtime

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

# ViT model
checkpoint = "facebook/deit-tiny-patch16-224"
image_processor = AutoImageProcessor.from_pretrained(checkpoint)

# Initialize the ViT model using Hugging Face AutoModelForImageClassification
model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

In [ ]:
# Apply ViT-specific transforms to the already split 20-class 'food' dataset.
food_vit = food.copy()

food_vit["train"] = food_vit["train"].with_transform(
    lambda examples: apply_transforms(examples, image_processor, is_train=True)
)

food_vit["test"] = food_vit["test"].with_transform(
    lambda examples: apply_transforms(examples, image_processor, is_train=False)
)

print("ViT-specific image transforms applied to the 20-class Food101 subset.")

# Sanity check: the Trainer should receive tensors, not raw PIL images.
sample = food_vit["train"][0]
print("\nTransformed sample keys:", sample.keys())
print("pixel_values type:", type(sample["pixel_values"]))
print("pixel_values shape:", sample["pixel_values"].shape)
print("label:", sample["label"], "->", id2label[int(sample["label"])])

assert "image" not in sample, "Raw image column is still present. This will break DefaultDataCollator."
assert "pixel_values" in sample, "pixel_values missing. Transform was not applied correctly."


### Freezing Backbone Layers

To fine-tune only the classification head, we need to freeze the parameters of the model's feature extractor (the backbone). This means we'll set `requires_grad=False` for all parameters except those belonging to the `model.head` module.

In [ ]:
# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the classification head (typically 'classifier' or 'head' in ViT models)
for param in model.classifier.parameters():
    param.requires_grad = True

print("Model parameters frozen. Only classification head will be fine-tuned.")
print(model)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/food101_20cls_250_head_only",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    warmup_steps=10,
    logging_steps=10,
    run_name="food101",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=food_vit["train"],
    eval_dataset=food_vit["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)


print("Starting ViT Head-Only fine-tuning...")
vit_head_only_train_time = conditional_train_model(model, trainer, training_args, "ViT Head-Only Fine-tune")
print("ViT Head-Only fine-tuning complete.")


## Fine-tuning the Whole Model (All Layers Unfrozen)

This section fine-tunes the entire Vision Transformer model (all layers, not just the classification head) on the Food101 dataset.

In [ ]:
# Re-initialize the model with all layers unfrozen
model_full_finetune = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# Ensure all parameters require gradients
for param in model_full_finetune.parameters():
    param.requires_grad = True

print("Model re-initialized. All parameters are unfrozen and will be trained.")
print(model_full_finetune)

In [ ]:
optimizer_grouped_parameters = [
    {"params": model_full_finetune.vit.parameters(), "lr": 2e-5},
    {"params": model_full_finetune.classifier.parameters(), "lr": 1e-4},
]

training_args_full = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/food101_20cls_250_full_finetune",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    warmup_steps=10,
    logging_steps=10,
    run_name="food101_full_finetune",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer_full = Trainer(
    model=model_full_finetune,
    args=training_args_full,
    data_collator=data_collator,
    train_dataset=food_vit["train"],
    eval_dataset=food_vit["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
    optimizers=(torch.optim.AdamW(optimizer_grouped_parameters), None),
)

print("Starting full model fine-tuning...")
vit_full_train_time = conditional_train_model(model_full_finetune, trainer_full, training_args_full, "ViT Full Fine-tune")
print("Full model fine-tuning complete.")


## LoRA Fine-tuning

This section demonstrates fine-tuning the Vision Transformer model using Low-Rank Adaptation (LoRA) to reduce computational cost and memory footprint during training.

In [ ]:
from peft import LoraConfig, get_peft_model

# Define LoRA configuration
lora_config_dict = {
    "r": 8,  # LoRA attention dimension
    "lora_alpha": 16,  # Alpha parameter for LoRA scaling
    "target_modules": ["q_proj", "v_proj"], # Correct target modules for LoRA in ViT
    "lora_dropout": 0.05,  # Dropout probability
    "bias": "none",  # Bias type
    "task_type": "SEQ_CLS" # Task type for image classification
}

# Re-initialize the base model
model_lora_base = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

lora_config = LoraConfig(**lora_config_dict)

# Wrap the base model with LoRA
model_lora = get_peft_model(model_lora_base, lora_config)

print("LoRA model created:")
model_lora.print_trainable_parameters()
print(model_lora)

In [ ]:
# Set up TrainingArguments for LoRA fine-tuning
training_args_lora = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/food101_20cls_250_lora",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    warmup_steps=10,
    logging_steps=10,
    run_name="food101_lora_finetune",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    data_collator=data_collator,
    train_dataset=food_vit["train"],
    eval_dataset=food_vit["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

print("Starting LoRA fine-tuning...")
vit_lora_train_time = conditional_train_model(model_lora, trainer_lora, training_args_lora, "ViT LoRA Fine-tune")
print("LoRA fine-tuning complete.")


## Evaluate Head-Only Fine-tuned Model

Using the `evaluate_and_plot` function to assess the performance of the head-only fine-tuned model on the Food101 test set, including accuracy, F1-score, and a confusion matrix. This section also prints the number of trainable parameters for this stage.

In [ ]:
print("Trainable parameters for Head-Only Fine-tuning:")
count_parameters(model)

# Evaluate the head-only fine-tuned model on Food101 test set
head_only_metrics_food101 = evaluate_and_plot(
    model=model,
    trainer=trainer,
    dataset=food_vit["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (Head-Only)",
    num_labels=len(id2label),
    model_display_name="ViT Head-Only Fine-tune",
    training_time=vit_head_only_train_time
)

## Evaluate Full Model Fine-tuned Model

Using the `evaluate_and_plot` function to assess the performance of the full model fine-tuned model on the Food101 test set, including accuracy, F1-score, and a confusion matrix. This section also prints the number of trainable parameters for this stage.

In [ ]:
print("Trainable parameters for Full Model Fine-tuning:")
count_parameters(model_full_finetune)

# Evaluate the full model fine-tuned model on Food101 test set
full_finetune_metrics_food101 = evaluate_and_plot(
    model=model_full_finetune,
    trainer=trainer_full,
    dataset=food_vit["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (Full Fine-tune)",
    num_labels=len(id2label),
    model_display_name="ViT Full Fine-tune",
    training_time=vit_full_train_time
)

## Evaluate LoRA Fine-tuned Model

Using the `evaluate_and_plot` function to assess the performance of the LoRA fine-tuned model on the Food101 test set, including accuracy, F1-score, and a confusion matrix. This section also prints the number of trainable parameters for this stage.

In [ ]:
print("Trainable parameters for LoRA Fine-tuning:")
model_lora.print_trainable_parameters() # LoRA models have their own method for this

# Evaluate the LoRA fine-tuned model on Food101 test set
lora_metrics_food101 = evaluate_and_plot(
    model=model_lora,
    trainer=trainer_lora,
    dataset=food_vit["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (LoRA)",
    num_labels=len(id2label),
    model_display_name="ViT LoRA Fine-tune",
    training_time=vit_lora_train_time
)

## Fine-tuning Last 3 Encoder Blocks + Head

This section will fine-tune the classification head along with the last 3 encoder blocks of the ViT backbone on the Food101 dataset.

In [ ]:
# Re-initialize the model and freeze all layers initially
model_last_3_blocks = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# Freeze all layers
for param in model_last_3_blocks.parameters():
    param.requires_grad = False

# Unfreeze the classification head
for param in model_last_3_blocks.classifier.parameters():
    param.requires_grad = True

# Unfreeze the last 3 encoder layers
num_encoder_layers = len(model_last_3_blocks.vit.layers)
for i in range(num_encoder_layers - 3, num_encoder_layers):
    for param in model_last_3_blocks.vit.layers[i].parameters():
        param.requires_grad = True

print("Model re-initialized. Last 3 encoder blocks and head are unfrozen for fine-tuning.")
count_parameters(model_last_3_blocks)

### Train the Model (Last 3 Blocks + Head)

In [ ]:
training_args_last_3_blocks = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/food101_20cls_250_last_3_blocks",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    warmup_steps=10,
    logging_steps=10,
    run_name="food101_last_3_blocks",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer_last_3_blocks = Trainer(
    model=model_last_3_blocks,
    args=training_args_last_3_blocks,
    data_collator=data_collator,
    train_dataset=food_vit["train"],
    eval_dataset=food_vit["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning for last 3 blocks + head...")
vit_last_3_blocks_train_time = conditional_train_model(model_last_3_blocks, trainer_last_3_blocks, training_args_last_3_blocks, "ViT Last 3 Blocks + Head Fine-tune")
print("Fine-tuning for last 3 blocks + head complete.")


### Evaluate Model (Last 3 Blocks + Head)

In [ ]:
print("Trainable parameters for Last 3 Blocks + Head Fine-tuning:")
count_parameters(model_last_3_blocks)

last_3_blocks_metrics_food101 = evaluate_and_plot(
    model=model_last_3_blocks,
    trainer=trainer_last_3_blocks,
    dataset=food_vit["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (Last 3 Blocks + Head)",
    num_labels=len(id2label),
    model_display_name="ViT Last 3 Blocks + Head Fine-tune",
    training_time=vit_last_3_blocks_train_time
)

## Fine-tuning Last 5 Encoder Blocks + Head

This section will fine-tune the classification head along with the last 5 encoder blocks of the ViT backbone on the Food101 dataset.

In [ ]:
# Re-initialize the model and freeze all layers initially
model_last_5_blocks = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# Freeze all layers
for param in model_last_5_blocks.parameters():
    param.requires_grad = False

# Unfreeze the classification head
for param in model_last_5_blocks.classifier.parameters():
    param.requires_grad = True

# Unfreeze the last 5 encoder layers
num_encoder_layers = len(model_last_5_blocks.vit.layers)
for i in range(num_encoder_layers - 5, num_encoder_layers):
    for param in model_last_5_blocks.vit.layers[i].parameters():
        param.requires_grad = True

print("Model re-initialized. Last 5 encoder blocks and head are unfrozen for fine-tuning.")
count_parameters(model_last_5_blocks)

### Train the Model (Last 5 Blocks + Head)

In [ ]:
training_args_last_5_blocks = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/food101_20cls_250_last_5_blocks",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    warmup_steps=10,
    logging_steps=10,
    run_name="food101_last_5_blocks",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer_last_5_blocks = Trainer(
    model=model_last_5_blocks,
    args=training_args_last_5_blocks,
    data_collator=data_collator,
    train_dataset=food_vit["train"],
    eval_dataset=food_vit["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning for last 5 blocks + head...")
vit_last_5_blocks_train_time = conditional_train_model(model_last_5_blocks, trainer_last_5_blocks, training_args_last_5_blocks, "ViT Last 5 Blocks + Head Fine-tune")
print("Fine-tuning for last 5 blocks + head complete.")


### Evaluate Model (Last 5 Blocks + Head)

In [ ]:
print("Trainable parameters for Last 5 Blocks + Head Fine-tuning:")
count_parameters(model_last_5_blocks)

last_5_blocks_metrics_food101 = evaluate_and_plot(
    model=model_last_5_blocks,
    trainer=trainer_last_5_blocks,
    dataset=food_vit["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (Last 5 Blocks + Head)",
    num_labels=len(id2label),
    model_display_name="ViT Last 5 Blocks + Head Fine-tune",
    training_time=vit_last_5_blocks_train_time
)

## ResNet-50 Full Model Fine-tuning (using `transformers`)

This section will fine-tune a ResNet-50 model using the `AutoImageProcessor` and `AutoModelForImageClassification` from Hugging Face `transformers`, ensuring appropriate preprocessing and consistent training parameters.

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

# ResNet-50 Specific Preprocessing
resnet_model_id_hf = "microsoft/resnet-50"

# Load the image processor
resnet_image_processor = AutoImageProcessor.from_pretrained(resnet_model_id_hf)

# Apply ResNet-specific transforms to the already split 'food' dataset using the unified function
food_resnet = food.copy() # Create a copy to avoid modifying the original 'food' dataset directly
food_resnet["train"] = food_resnet["train"].with_transform(lambda examples: apply_transforms(examples, resnet_image_processor, is_train=True))
food_resnet["test"] = food_resnet["test"].with_transform(lambda examples: apply_transforms(examples, resnet_image_processor, is_train=False))

print("ResNet-50 specific image processor and transforms defined and applied to the shared dataset split using the unified function.")

In [ ]:
# Load the ResNet-50 model from Hugging Face Transformers
resnet_model_hf = AutoModelForImageClassification.from_pretrained(
    resnet_model_id_hf,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True  # Set this to True to ignore mismatched sizes
)

# Ensure all parameters require gradients for full fine-tuning
for param in resnet_model_hf.parameters():
    param.requires_grad = True

print("ResNet-50 model loaded and configured for full fine-tuning.")
print(resnet_model_hf)

In [ ]:
from transformers import TrainingArguments, Trainer

# Define TrainingArguments for ResNet-50
training_args_resnet_hf = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/food101_20cls_250_resnet50_full_finetune",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=training_args_full.per_device_train_batch_size,
    gradient_accumulation_steps=training_args_full.gradient_accumulation_steps,
    per_device_eval_batch_size=training_args_full.per_device_eval_batch_size,
    num_train_epochs=training_args_full.num_train_epochs,
    warmup_steps=training_args_full.warmup_steps,
    logging_steps=training_args_full.logging_steps,
    run_name="food101_resnet50_hf_full_finetune",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

# Initialize the Trainer for ResNet-50
trainer_resnet_hf = Trainer(
    model=resnet_model_hf,
    args=training_args_resnet_hf,
    data_collator=data_collator,
    train_dataset=food_resnet["train"],
    eval_dataset=food_resnet["test"],
    processing_class=resnet_image_processor,
    compute_metrics=compute_metrics,
)

print("Starting ResNet-50 (Hugging Face) full model fine-tuning...")
resnet_hf_full_train_time = conditional_train_model(resnet_model_hf, trainer_resnet_hf, training_args_resnet_hf, "ResNet-50 HF Full Fine-tune")
print("ResNet-50 (Hugging Face) full model fine-tuning complete.")


### Evaluate ResNet-50 (Hugging Face) Full Fine-tuned Model

In [ ]:
print("Trainable parameters for ResNet-50 (Hugging Face) Full Model Fine-tuning:")
count_parameters(resnet_model_hf)

resnet_hf_full_metrics_food101 = evaluate_and_plot(
    model=resnet_model_hf,
    trainer=trainer_resnet_hf,
    dataset=food_resnet["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (ResNet-50 HF Full Fine-tune)",
    num_labels=len(id2label),
    model_display_name="ResNet-50 HF Full Fine-tune",
    training_time=resnet_hf_full_train_time
)

## Summary of Fine-tuning Experiment Results on Food101

In [ ]:
import pandas as pd

results_data = [
    {
        "Strategy": "ViT Head-Only Fine-tune",
        "Trainable Parameters (millions)": round(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 2),
        "Number of Epochs": training_args.num_train_epochs,
        "Learning Rate": training_args.learning_rate,
        "Training Time (seconds)": vit_head_only_train_time,
        "Accuracy": head_only_metrics_food101["accuracy"],
        "Macro F1": head_only_metrics_food101["macro_f1"],
        "Weighted F1": head_only_metrics_food101["weighted_f1"],
        "Top-5 Accuracy": head_only_metrics_food101["top_5_accuracy"],
        "Eval Loss": head_only_metrics_food101.get("eval_loss", None),
        "Mean Confidence": head_only_metrics_food101["mean_top1_confidence"],
        "Mean Margin": head_only_metrics_food101["mean_top1_top2_margin"],
        "ECE": head_only_metrics_food101["ece"],
        "NLL": head_only_metrics_food101["nll"],
        "Brier Score": head_only_metrics_food101["brier_score"],
    },
    {
        "Strategy": "ViT Full Fine-tune",
        "Trainable Parameters (millions)": round(sum(p.numel() for p in model_full_finetune.parameters() if p.requires_grad) / 1e6, 2),
        "Number of Epochs": training_args_full.num_train_epochs,
        "Learning Rate": training_args_full.learning_rate,
        "Training Time (seconds)": vit_full_train_time,
        "Accuracy": full_finetune_metrics_food101["accuracy"],
        "Macro F1": full_finetune_metrics_food101["macro_f1"],
        "Weighted F1": full_finetune_metrics_food101["weighted_f1"],
        "Top-5 Accuracy": full_finetune_metrics_food101["top_5_accuracy"],
        "Eval Loss": full_finetune_metrics_food101.get("eval_loss", None),
        "Mean Confidence": full_finetune_metrics_food101["mean_top1_confidence"],
        "Mean Margin": full_finetune_metrics_food101["mean_top1_top2_margin"],
        "ECE": full_finetune_metrics_food101["ece"],
        "NLL": full_finetune_metrics_food101["nll"],
        "Brier Score": full_finetune_metrics_food101["brier_score"],
    },
    {
        "Strategy": "ViT LoRA Fine-tune",
        "Trainable Parameters (millions)": round(sum(p.numel() for p in model_lora.parameters() if p.requires_grad) / 1e6, 2),
        "Number of Epochs": training_args_lora.num_train_epochs,
        "Learning Rate": training_args_lora.learning_rate,
        "Training Time (seconds)": vit_lora_train_time,
        "Accuracy": lora_metrics_food101["accuracy"],
        "Macro F1": lora_metrics_food101["macro_f1"],
        "Weighted F1": lora_metrics_food101["weighted_f1"],
        "Top-5 Accuracy": lora_metrics_food101["top_5_accuracy"],
        "Eval Loss": lora_metrics_food101.get("eval_loss", None),
        "Mean Confidence": lora_metrics_food101["mean_top1_confidence"],
        "Mean Margin": lora_metrics_food101["mean_top1_top2_margin"],
        "ECE": lora_metrics_food101["ece"],
        "NLL": lora_metrics_food101["nll"],
        "Brier Score": lora_metrics_food101["brier_score"],
    },
    {
        "Strategy": "ViT Last 3 Blocks + Head Fine-tune",
        "Trainable Parameters (millions)": round(sum(p.numel() for p in model_last_3_blocks.parameters() if p.requires_grad) / 1e6, 2),
        "Number of Epochs": training_args_last_3_blocks.num_train_epochs,
        "Learning Rate": training_args_last_3_blocks.learning_rate,
        "Training Time (seconds)": vit_last_3_blocks_train_time,
        "Accuracy": last_3_blocks_metrics_food101["accuracy"],
        "Macro F1": last_3_blocks_metrics_food101["macro_f1"],
        "Weighted F1": last_3_blocks_metrics_food101["weighted_f1"],
        "Top-5 Accuracy": last_3_blocks_metrics_food101["top_5_accuracy"],
        "Eval Loss": last_3_blocks_metrics_food101.get("eval_loss", None),
        "Mean Confidence": last_3_blocks_metrics_food101["mean_top1_confidence"],
        "Mean Margin": last_3_blocks_metrics_food101["mean_top1_top2_margin"],
        "ECE": last_3_blocks_metrics_food101["ece"],
        "NLL": last_3_blocks_metrics_food101["nll"],
        "Brier Score": last_3_blocks_metrics_food101["brier_score"],
    },
    {
        "Strategy": "ViT Last 5 Blocks + Head Fine-tune",
        "Trainable Parameters (millions)": round(sum(p.numel() for p in model_last_5_blocks.parameters() if p.requires_grad) / 1e6, 2),
        "Number of Epochs": training_args_last_5_blocks.num_train_epochs,
        "Learning Rate": training_args_last_5_blocks.learning_rate,
        "Training Time (seconds)": vit_last_5_blocks_train_time,
        "Accuracy": last_5_blocks_metrics_food101["accuracy"],
        "Macro F1": last_5_blocks_metrics_food101["macro_f1"],
        "Weighted F1": last_5_blocks_metrics_food101["weighted_f1"],
        "Top-5 Accuracy": last_5_blocks_metrics_food101["top_5_accuracy"],
        "Eval Loss": last_5_blocks_metrics_food101.get("eval_loss", None),
        "Mean Confidence": last_5_blocks_metrics_food101["mean_top1_confidence"],
        "Mean Margin": last_5_blocks_metrics_food101["mean_top1_top2_margin"],
        "ECE": last_5_blocks_metrics_food101["ece"],
        "NLL": last_5_blocks_metrics_food101["nll"],
        "Brier Score": last_5_blocks_metrics_food101["brier_score"],
    },
]

# Add ResNet if it has already been evaluated
if "resnet_hf_full_metrics_food101" in globals():
    results_data.append(
        {
            "Strategy": "ResNet-50 HF Full Fine-tune",
            "Trainable Parameters (millions)": round(sum(p.numel() for p in resnet_model_hf.parameters() if p.requires_grad) / 1e6, 2),
            "Number of Epochs": training_args_resnet_hf.num_train_epochs,
            "Learning Rate": training_args_resnet_hf.learning_rate,
            "Training Time (seconds)": resnet_hf_full_train_time,
            "Accuracy": resnet_hf_full_metrics_food101["accuracy"],
            "Macro F1": resnet_hf_full_metrics_food101["macro_f1"],
            "Weighted F1": resnet_hf_full_metrics_food101["weighted_f1"],
            "Top-5 Accuracy": resnet_hf_full_metrics_food101["top_5_accuracy"],
            "Eval Loss": resnet_hf_full_metrics_food101.get("eval_loss", None),
            "Mean Confidence": resnet_hf_full_metrics_food101["mean_top1_confidence"],
            "Mean Margin": resnet_hf_full_metrics_food101["mean_top1_top2_margin"],
            "ECE": resnet_hf_full_metrics_food101["ece"],
            "NLL": resnet_hf_full_metrics_food101["nll"],
            "Brier Score": resnet_hf_full_metrics_food101["brier_score"],
        }
    )

results_df = pd.DataFrame(results_data)
results_df = results_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
display(results_df.round(4))